In [94]:
function LU_IR(A, b; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    # Step 1: factorization in ul precision
    LU = lu(ul.(A))

    # Step 2: solve x in ul precision
    x = LU \ ul.(b) # should I keep x in higher precision after the operation? 

    for i in 1:imax
        # Step 3: compute residual in ur precision
        r = ur.(b) - ur.(A) * ur.(x)

        # Step 4: solve d in ul precision
        d = LU \ ul.(r)
        
        # Step 5: update x in u precision
        x = u.(x) + u.(d)
        
        if abs(norm(r)) < atol
            println("Converged at iter $(i)")
            break
        end
    end

    return x
end

LU_IR (generic function with 1 method)

In [96]:
function example(n, m; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    A = rand(n, m)
    b = rand(n)

    start_time = time()
    x_true = A \ b
    elapsed = time() - start_time
    x_true = vec(x_true)
    println("Run time: ", elapsed)

    start_time = time()
    x = LU_IR(A, b)
    elapsed = time() - start_time
    println("Run time: ", elapsed)
    
    error = abs(norm(x-x_true))
    println("Error: ", error)
    

end    

example (generic function with 1 method)

In [98]:
using LinearAlgebra
using BenchmarkTools

example(100, 100)

Run time: 0.001547098159790039
Run time: 0.000637054443359375
Error: 3.518359773143122e-6
